In [ ]:
!pip install langchain openai langchain-openai langchain-community


In [ ]:
OPENAI_API_KEY = "sk-XXXXXXXXXXXXXXXXXXXXXXXXXXXX"

In [ ]:
from dotenv import load_dotenv
import os

In [ ]:
%load_ext dotenv
%dotenv

In [ ]:
from langchain_openai import ChatOpenAI


In [ ]:
llm = ChatOpenAI(model_name="gpt-5.4-mini", api_key=os.getenv("OPENAI_API_KEY"))

In [ ]:
llm.invoke("Tell me a funny joke")

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

In [ ]:
messages = [
    SystemMessage(content="You are an AI assistant designed to tell funny jokes. Do not answer any questions that are not related to telling jokes."),
    HumanMessage(content="Tell me a funny joke."),
]

In [ ]:
response = llm.invoke(messages)
response.content

In [ ]:
from langchain_core.prompts import PromptTemplate

In [ ]:
prompt_template = PromptTemplate(name="Historic Fact Prompt", template="Tell me a fact about a historic fact about {event} in {location}.", input_variables=["event", "location"])

In [ ]:
prompt_template.format(event="the moon landing", location="the United States")

In [ ]:
from langchain_core.prompts import (
    PromptTemplate,
    ChatPromptTemplate,
    HumanMessagePromptTemplate,
    SystemMessagePromptTemplate
)

In [ ]:
system_message_str = """
You are a helpful AI assistant. Given the 
following context, answer the question. If the answer can not be 
found in the context, simply say you DO NOT KNOW.
Context: {context}
"""

system_message_prompt = SystemMessagePromptTemplate(
    prompt=PromptTemplate(
        template=system_message_str, 
        input_variables=["context"]
        )
    )

In [ ]:
human_message_str = "Can you provide details on: {question}?"

human_message_prompt = HumanMessagePromptTemplate(
    prompt=PromptTemplate(
        input_variables=["question"],
        template=human_message_str
    )
)

In [ ]:
messages = [system_message_prompt, human_message_prompt]

chatbot_prompt_template = ChatPromptTemplate(
    messages=messages,
    input_variables=["context", "question"]
)

In [ ]:
question = "What it the capital city of Kenya?"
context = "Nairobi is the capital city of Kenya."

chatbot_prompt_template.format(context=context, question=question)

llm.invoke(chatbot_prompt_template.format(context=context, question=question))

In [ ]:
from langchain_core.output_parsers import StrOutputParser


chain = chatbot_prompt_template | llm | StrOutputParser()

In [ ]:
chain.invoke({"context": context, "question": question})

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.tools import ToolException
from typing import Literal, Optional


In [66]:
operator_type = Literal["add", "subtract", "multiply", "divide"]

In [67]:
@tool("Calculator", return_direct=True)
def calculator(operator: operator_type, num1: float, num2: float) -> float:
    """
    A simple calculator tool.
    """
    if operator == "add":
        return num1 + num2
    elif operator == "subtract":
        return num1 - num2
    elif operator == "multiply":
        return num1 * num2
    elif operator == "divide":
        if num2 == 0:
            raise ToolException("Cannot divide by zero.")
        return num1 / num2
    else:
        raise ToolException(f"Invalid operator: {operator}")


In [68]:
print(calculator.name)
print(calculator.description)
print(calculator.args_schema)

Calculator
A simple calculator tool.
<class 'langchain_core.utils.pydantic.Calculator'>


In [69]:
calculator.run({"operator": "add", "num1": 10, "num2": 5})

15.0

In [70]:
tools = [calculator]

In [71]:
system_prompt = "You are a helpful assistant. Use the tools available to you when they apply."


In [72]:
agent = create_agent(model=llm, tools=tools, system_prompt=system_prompt)


In [73]:
response = agent.invoke(
    {
        "messages": [{"role": "user", "content": "What is the result of adding 10 and 5?"}]
    }
)

response["messages"][-1].content


'15.0'